In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/rongardF/tvdatafeed.git pandas numpy -q

import pandas as pd
import numpy as np
import time
from datetime import datetime
from zoneinfo import ZoneInfo
from tvDatafeed import TvDatafeed, Interval

SYMBOL = "XAUUSD"
EXCHANGE = "OANDA"

BUFFER = 0.20
RR_CHOICES = [4, 3, 2]

NORMAL_MIN_MC = 50
SCALP_MIN_MC = 45

BARS_5M = 500
BARS_15M = 300
BARS_1H = 300
BARS_4H = 300

MC_SIMULATIONS = 500
MC_STEPS = 12

tv = TvDatafeed()

STATE_PARAMS = {
    "LAMINAR_BULL": {"drift": 0.18, "vol": 0.90},
    "LAMINAR_BEAR": {"drift": -0.18, "vol": 0.90},
    "TURBULENT_BULL": {"drift": 0.08, "vol": 1.60},
    "TURBULENT_BEAR": {"drift": -0.08, "vol": 1.60},
    "TRANSITION_BULL": {"drift": 0.10, "vol": 1.20},
    "TRANSITION_BEAR": {"drift": -0.10, "vol": 1.20},
    "CHOP": {"drift": 0.00, "vol": 1.10},
}

def ema(s, n):
    return s.ewm(span=n, adjust=False).mean()

def atr(df, n=14):
    pc = df["close"].shift(1)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - pc).abs(),
        (df["low"] - pc).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(n).mean()

def rsi(s, n=14):
    delta = s.diff()
    gain = delta.clip(lower=0).rolling(n).mean()
    loss = (-delta.clip(upper=0)).rolling(n).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def macd(df):
    fast = ema(df["close"], 12)
    slow = ema(df["close"], 26)
    line = fast - slow
    signal = ema(line, 9)
    hist = line - signal
    return line, signal, hist

def bollinger(df, n=20, k=2):
    mid = df["close"].rolling(n).mean()
    std = df["close"].rolling(n).std()
    upper = mid + k * std
    lower = mid - k * std
    width = (upper - lower) / mid.replace(0, np.nan)
    return upper, mid, lower, width

def donchian(df, n=20):
    return df["high"].rolling(n).max(), df["low"].rolling(n).min()

def laplacian(s):
    return s - 2 * s.shift(1) + s.shift(2)

def fetch_tv(interval, bars):
    df = tv.get_hist(symbol=SYMBOL, exchange=EXCHANGE, interval=interval, n_bars=bars)
    if df is None or df.empty:
        raise ValueError("TradingView OANDA feed returned no data")

    df = df.copy()
    df.columns = [str(c).lower() for c in df.columns]

    if "volume" not in df.columns:
        df["volume"] = 0

    return df[["open", "high", "low", "close", "volume"]].dropna()

def prepare_df(df):
    df = df.copy()

    df["EMA9"] = ema(df["close"], 9)
    df["EMA21"] = ema(df["close"], 21)
    df["EMA50"] = ema(df["close"], 50)
    df["ATR"] = atr(df)
    df["RSI"] = rsi(df["close"])
    df["LAP"] = laplacian(df["close"])

    _, _, macd_hist = macd(df)
    df["MACD_HIST"] = macd_hist
    df["MACD_HIST_CHANGE"] = df["MACD_HIST"].diff()

    _, bb_mid, _, bb_width = bollinger(df)
    df["BB_MID"] = bb_mid
    df["BB_WIDTH"] = bb_width
    df["BB_WIDTH_MA"] = df["BB_WIDTH"].rolling(50).mean()

    don_high, don_low = donchian(df)
    df["DONCHIAN_HIGH"] = don_high
    df["DONCHIAN_LOW"] = don_low

    df["LAMINAR"] = abs(df["EMA9"] - df["EMA21"]) / df["ATR"].replace(0, np.nan)
    df["ATR_MA"] = df["ATR"].rolling(50).mean()
    df["ATR_RATIO"] = df["ATR"] / df["ATR_MA"].replace(0, np.nan)

    return df.replace([np.inf, -np.inf], np.nan).dropna()

def get_bias(df):
    row = df.iloc[-1]

    bull = 0
    bear = 0

    bull += int(row["EMA9"] > row["EMA21"])
    bear += int(row["EMA9"] < row["EMA21"])

    bull += int(row["close"] > row["EMA50"])
    bear += int(row["close"] < row["EMA50"])

    bull += int(row["MACD_HIST"] > 0)
    bear += int(row["MACD_HIST"] < 0)

    bull += int(row["RSI"] > 50)
    bear += int(row["RSI"] < 50)

    if bull >= 3:
        return "BULLISH"
    elif bear >= 3:
        return "BEARISH"
    return "NEUTRAL"

def get_context_4h(df):
    row = df.iloc[-1]

    if row["ATR_RATIO"] > 2.50:
        return "TOO_VOLATILE"

    if row["BB_WIDTH"] < row["BB_WIDTH_MA"] or row["ATR_RATIO"] < 0.60:
        return "CONSOLIDATING"

    return "TRADABLE"

def get_ny_time_and_session():
    ny = datetime.now(ZoneInfo("America/New_York"))
    ist = datetime.now(ZoneInfo("Asia/Kolkata"))

    h_ny = ny.hour + ny.minute / 60
    h_ist = ist.hour + ist.minute / 60

    if 18.5 <= h_ist <= 20:
        return ny, 1.35, "SCALP_WINDOW"

    if 7 <= h_ny < 11:
        return ny, 1.25, "NY_POWER"
    elif 11 <= h_ny < 13:
        return ny, 0.90, "NY_MIDDAY"
    elif 13 <= h_ny < 16:
        return ny, 1.05, "NY_AFTERNOON"

    return ny, 0.0, "NO_TRADE_SESSION"

def get_zones(df_15m, df_1h, price):
    recent_15 = df_15m.tail(48)
    recent_1h = df_1h.tail(48)

    atr_15 = float(df_15m.iloc[-1]["ATR"])
    atr_1h = float(df_1h.iloc[-1]["ATR"])

    d_low_15 = float(recent_15["low"].min())
    d_high_15 = d_low_15 + atr_15 * 0.50

    s_high_15 = float(recent_15["high"].max())
    s_low_15 = s_high_15 - atr_15 * 0.50

    d_low_1h = float(recent_1h["low"].min())
    d_high_1h = d_low_1h + atr_1h * 0.35

    s_high_1h = float(recent_1h["high"].max())
    s_low_1h = s_high_1h - atr_1h * 0.35

    demands = [("15M", d_low_15, d_high_15), ("1H", d_low_1h, d_high_1h)]
    supplies = [("15M", s_low_15, s_high_15), ("1H", s_low_1h, s_high_1h)]

    valid_demands = [z for z in demands if z[2] < price]
    valid_supplies = [z for z in supplies if z[1] > price]

    demand = max(valid_demands, key=lambda z: z[2]) if valid_demands else demands[0]
    supply = min(valid_supplies, key=lambda z: z[1]) if valid_supplies else supplies[0]

    d_tf, d_low, d_high = demand
    s_tf, s_low, s_high = supply

    d_width = max(d_high - d_low, 0.01)
    s_width = max(s_high - s_low, 0.01)

    return {
        "demand_tf": d_tf,
        "d_buf_low": d_low - BUFFER * d_width,
        "d_buf_high": d_high + BUFFER * d_width,
        "supply_tf": s_tf,
        "s_buf_low": s_low - BUFFER * s_width,
        "s_buf_high": s_high + BUFFER * s_width
    }

def get_market_state(row):
    atr_ratio = float(row["ATR_RATIO"])
    laminar = float(row["LAMINAR"]) > 0.05

    ema_bull = row["EMA9"] > row["EMA21"]
    ema_bear = row["EMA9"] < row["EMA21"]

    lap_bull = row["LAP"] > 0
    lap_bear = row["LAP"] < 0

    if laminar and ema_bull and lap_bull:
        return "LAMINAR_BULL"
    if laminar and ema_bear and lap_bear:
        return "LAMINAR_BEAR"

    if atr_ratio > 1.80 and ema_bull:
        return "TURBULENT_BULL"
    if atr_ratio > 1.80 and ema_bear:
        return "TURBULENT_BEAR"

    if ema_bull or lap_bull:
        return "TRANSITION_BULL"
    if ema_bear or lap_bear:
        return "TRANSITION_BEAR"

    return "CHOP"

def monte_carlo_probability(entry, sl, tp, state, atr_now, simulations=MC_SIMULATIONS, steps=MC_STEPS):
    params = STATE_PARAMS.get(state, STATE_PARAMS["CHOP"])

    drift = params["drift"] * atr_now
    vol = params["vol"] * atr_now

    tp_hits = 0
    sl_hits = 0

    direction = "BUY" if tp > entry else "SELL"

    for _ in range(simulations):
        sim_price = entry

        for _ in range(steps):
            move = np.random.normal(loc=drift, scale=vol)
            sim_price += move

            if direction == "BUY":
                if sim_price >= tp:
                    tp_hits += 1
                    break
                if sim_price <= sl:
                    sl_hits += 1
                    break

            else:
                if sim_price <= tp:
                    tp_hits += 1
                    break
                if sim_price >= sl:
                    sl_hits += 1
                    break

    total = tp_hits + sl_hits

    if total == 0:
        return 0.0

    return round((tp_hits / total) * 100, 2)

def choose_best_rr(direction, entry, sl, zones, state, atr_now):
    best = None

    for rr in RR_CHOICES:
        risk = abs(entry - sl)

        if risk <= 0:
            continue

        if direction == "BUY":
            tp = entry + rr * risk
            structure_valid = tp <= zones["s_buf_low"]
        else:
            tp = entry - rr * risk
            structure_valid = tp >= zones["d_buf_high"]

        if not structure_valid:
            continue

        mc_prob = monte_carlo_probability(entry, sl, tp, state, atr_now)

        expectancy = (mc_prob / 100.0) * rr - (1 - mc_prob / 100.0)

        if best is None or expectancy > best["expectancy"]:
            best = {
                "rr": rr,
                "tp": tp,
                "mc_prob": mc_prob,
                "expectancy": expectancy
            }

    return best

def signal_once():
    df_5m = prepare_df(fetch_tv(Interval.in_5_minute, BARS_5M))
    df_15m = prepare_df(fetch_tv(Interval.in_15_minute, BARS_15M))
    df_1h = prepare_df(fetch_tv(Interval.in_1_hour, BARS_1H))
    df_4h = prepare_df(fetch_tv(Interval.in_4_hour, BARS_4H))

    bias_1h = get_bias(df_1h)
    bias_4h = get_bias(df_4h)
    context_4h = get_context_4h(df_4h)

    row = df_5m.iloc[-1]
    prev = df_5m.iloc[-2]

    candle_time = str(df_5m.index[-1])
    price = float(row["close"])
    atr_now = float(row["ATR"])

    ny_time, session_mult, session_name = get_ny_time_and_session()

    zones = get_zones(df_15m, df_1h, price)

    near_demand = zones["d_buf_low"] <= price <= zones["d_buf_high"]
    near_supply = zones["s_buf_low"] <= price <= zones["s_buf_high"]

    bull_reclaim = bool(prev["low"] <= prev["DONCHIAN_LOW"] and price > row["DONCHIAN_LOW"])
    bear_reject = bool(prev["high"] >= prev["DONCHIAN_HIGH"] and price < row["DONCHIAN_HIGH"])

    atr_ok = bool(0.50 <= float(row["ATR_RATIO"]) <= 2.50)

    macd_bull = bool(row["MACD_HIST"] > 0 or row["MACD_HIST_CHANGE"] > 0)
    macd_bear = bool(row["MACD_HIST"] < 0 or row["MACD_HIST_CHANGE"] < 0)

    rsi_buy_ok = bool(32 <= float(row["RSI"]) <= 75)
    rsi_sell_ok = bool(25 <= float(row["RSI"]) <= 68)

    ema_bull = bool(row["EMA9"] > row["EMA21"])
    ema_bear = bool(row["EMA9"] < row["EMA21"])

    lap_bull = bool(row["LAP"] > 0)
    lap_bear = bool(row["LAP"] < 0)

    bb_bull = bool(price > row["BB_MID"])
    bb_bear = bool(price < row["BB_MID"])

    laminar_ok = bool(row["LAMINAR"] > 0.035)

    tube_width = max(zones["s_buf_low"] - zones["d_buf_high"], 0.01)
    distance_to_supply = max(zones["s_buf_low"] - price, 0.01)
    distance_to_demand = max(price - zones["d_buf_high"], 0.01)

    velocity = abs(float(df_5m["close"].diff().iloc[-1]))
    k_laminar = max(float(row["LAMINAR"]), 0.01)

    graetz_long = (velocity * tube_width ** 2) / (k_laminar * distance_to_supply)
    graetz_short = (velocity * tube_width ** 2) / (k_laminar * distance_to_demand)

    pullback_long = bool(
        bias_1h == "BULLISH"
        and price > row["EMA21"]
        and price <= row["EMA9"] + atr_now * 0.60
    )

    pullback_short = bool(
        bias_1h == "BEARISH"
        and price < row["EMA21"]
        and price >= row["EMA9"] - atr_now * 0.60
    )

    setup_long = near_demand or bull_reclaim or pullback_long
    setup_short = near_supply or bear_reject or pullback_short

    buy_score = sum([
        ema_bull, lap_bull, laminar_ok, setup_long,
        graetz_long > graetz_short, macd_bull,
        rsi_buy_ok, atr_ok, bb_bull, bias_1h == "BULLISH"
    ])

    sell_score = sum([
        ema_bear, lap_bear, laminar_ok, setup_short,
        graetz_short > graetz_long, macd_bear,
        rsi_sell_ok, atr_ok, bb_bear, bias_1h == "BEARISH"
    ])

    if bias_4h == "BULLISH":
        buy_score += 0.5
    elif bias_4h == "BEARISH":
        sell_score += 0.5

    buy_score *= session_mult
    sell_score *= session_mult

    state = get_market_state(row)

    signal = "NO TRADE"
    success = None
    entry = None
    sl = None
    tp = None
    rr_used = None
    exit_rule = "WAIT"

    min_mc = SCALP_MIN_MC if session_name == "SCALP_WINDOW" else NORMAL_MIN_MC

    if session_mult != 0 and atr_ok and context_4h != "TOO_VOLATILE":

        if bias_1h == "BULLISH" and setup_long and buy_score >= sell_score:
            test_entry = price
            test_sl = zones["d_buf_low"]

            if test_entry > test_sl:
                best = choose_best_rr("BUY", test_entry, test_sl, zones, state, atr_now)

                if best is not None and best["mc_prob"] >= min_mc:
                    signal = "BUY"
                    success = best["mc_prob"]
                    entry = test_entry
                    sl = test_sl
                    tp = best["tp"]
                    rr_used = best["rr"]
                    exit_rule = "EXIT AT TP OR SL"

        elif bias_1h == "BEARISH" and setup_short and sell_score >= buy_score:
            test_entry = price
            test_sl = zones["s_buf_high"]

            if test_entry < test_sl:
                best = choose_best_rr("SELL", test_entry, test_sl, zones, state, atr_now)

                if best is not None and best["mc_prob"] >= min_mc:
                    signal = "SELL"
                    success = best["mc_prob"]
                    entry = test_entry
                    sl = test_sl
                    tp = best["tp"]
                    rr_used = best["rr"]
                    exit_rule = "EXIT AT TP OR SL"

    return {
        "LIVE_TIME": ny_time.strftime("%Y-%m-%d %H:%M:%S"),
        "CANDLE_TIME": candle_time,
        "LIVE_PRICE": round(price, 2),
        "SIGNAL": signal,
        "BIAS_1H": bias_1h,
        "BIAS_4H": bias_4h,
        "CONTEXT_4H": context_4h,
        "STATE": state,
        "SUCCESS_PERCENT": success,
        "RR": None if rr_used is None else f"1:{rr_used}",
        "ENTRY": None if entry is None else round(entry, 2),
        "SL": None if sl is None else round(sl, 2),
        "TP": None if tp is None else round(tp, 2),
        "DEMAND_TF": zones["demand_tf"],
        "SUPPLY_TF": zones["supply_tf"],
        "EXIT": exit_rule
    }

last_candle_time = None
first_run = True

while True:
    try:
        sig = signal_once()

        if first_run or sig["CANDLE_TIME"] != last_candle_time:
            first_run = False
            last_candle_time = sig["CANDLE_TIME"]

            print("\n==============================")
            print("XAUUSD MONTE CARLO INDICATION")
            print("==============================")
            print(f"LIVE_TIME: {sig['LIVE_TIME']}")
            print(f"LIVE_PRICE: {sig['LIVE_PRICE']}")
            print(f"SIGNAL: {sig['SIGNAL']}")
            print(f"BIAS_1H: {sig['BIAS_1H']}")
            print(f"BIAS_4H: {sig['BIAS_4H']}")
            print(f"CONTEXT_4H: {sig['CONTEXT_4H']}")
            print(f"STATE: {sig['STATE']}")
            print(f"SUCCESS_PERCENT: {sig['SUCCESS_PERCENT']}")
            print(f"RR: {sig['RR']}")
            print(f"ENTRY: {sig['ENTRY']}")
            print(f"SL: {sig['SL']}")
            print(f"TP: {sig['TP']}")
            print(f"DEMAND_TF: {sig['DEMAND_TF']}")
            print(f"SUPPLY_TF: {sig['SUPPLY_TF']}")
            print(f"EXIT: {sig['EXIT']}")
            print("==============================\n")

        time.sleep(20)

    except Exception as e:
        print("\nERROR:", e)
        time.sleep(60)

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 126.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.4 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.



XAUUSD MONTE CARLO INDICATION
LIVE_TIME: 2026-06-29 07:00:48
LIVE_PRICE: 4037.53
SIGNAL: NO TRADE
BIAS_1H: BEARISH
BIAS_4H: NEUTRAL
CONTEXT_4H: CONSOLIDATING
STATE: TRANSITION_BULL
SUCCESS_PERCENT: None
RR: None
ENTRY: None
SL: None
TP: None
DEMAND_TF: 15M
SUPPLY_TF: 15M
EXIT: WAIT

